In [1]:
from plot_umap import *

/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [2]:
hs_train = pd.read_json("data/hellaswag_train_1ksubset.json")

In [3]:
# Encode context and endings
options = [0, 1, 2, 3]

data = {}
for id, entry in hs_train.iterrows():
    context = entry["ctx"]
    correct_ending = entry["endings"][entry["label"]]
    wrong_endings = [entry["endings"][i] for i in options if i != entry["label"]]

    data_point = {
        "text": context + " " + correct_ending,
        "correct": 1,
        "category": entry["activity_label"],
        "task_id": id
    }
    data[len(data)] = data_point
    for wrong in wrong_endings:
        data_point = {
            "text": context + " " + wrong,
            "correct": 0,
            "category": entry["activity_label"],
            "task_id": id
        }
        data[len(data)] = data_point

df = pd.DataFrame.from_dict(data, orient="index")
print("Total samples:", len(df))

Total samples: 4000


In [4]:
def filter_df(df: DataFrame, top_n_categories: int, n_samples: int) -> DataFrame:
    # Get top N categories
    top_n_categories = df['category'].value_counts().head(top_n_categories).index.tolist()
    filtered_df = df[df["category"].isin(top_n_categories)]

    # Group by task_id and sample groups rather than individual rows
    grouped = filtered_df.groupby('task_id')

    # Get list of all task_ids
    task_ids = list(grouped.groups.keys())

    # Sample task_ids (not individual rows)
    sampled_task_ids = pd.Series(task_ids).sample(n=min(round(n_samples / 4), len(task_ids)), random_state=42)

    # Get all rows for the sampled task_ids
    sampled_df = filtered_df[filtered_df['task_id'].isin(sampled_task_ids)]

    print(f"Number of samples={len(sampled_df)}, got {len(sampled_task_ids)} task groups")
    return sampled_df.reset_index(drop=True)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModel


def get_embeddings(df: DataFrame) -> Tensor:
    # Check if MPS is available
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load model and move to GPU
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("models/Mae_20251102-155319_1ksubset").to(device)
    model.train(False)

    inputs = tokenizer(df["text"].tolist(), padding=True, truncation=True, return_tensors="pt").to(
        device)  # Move all tensors to GPU

    # Generate embeddings in batches (faster and avoids OOM)
    batch_size = 64  # Adjust based on GPU memory
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(inputs["input_ids"]), batch_size):
            batch = {k: v[i:i + batch_size] for k, v in inputs.items()}
            batch_embeddings = model(**batch, return_dict=True).pooler_output
            embeddings.append(batch_embeddings.cpu())  # Move back to CPU

    # Concatenate all batches
    return torch.cat(embeddings, dim=0)

## Top 3 categories & 1'000 samples

In [6]:
df_top3_1000 = filter_df(df, 3, 1000)
embeddings_top_3_1000 = get_embeddings(df_top3_1000)

Number of samples=1000, got 250 task groups
Using device: mps


In [7]:
neighbors = list(range(18, 30, 2))
for neighbor in neighbors:
    plot_umap(embeddings_top_3_1000, df_top3_1000, neighbor)

/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## Single category & 100 samples

In [8]:
## Single category & 1'000 samples
df_top1_100 = filter_df(df, 1, 100)
embeddings_1_100 = get_embeddings(df_top1_100)

Number of samples=100, got 25 task groups
Using device: mps


In [9]:
neighbors = list(range(5, 30, 5))
for neighbor in neighbors:
    plot_umap(embeddings_1_100, df_top1_100, neighbor, color_label="task_id")

/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## Top 10 categories & 100 samples

In [10]:
df_top10_100 = filter_df(df, 10, 100)
embeddings_10_100 = get_embeddings(df_top10_100)

Number of samples=100, got 25 task groups
Using device: mps


In [11]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap(embeddings_10_100, df_top10_100, neighbor)

/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## Single category & 100 samples

In [12]:
df_top1_20 = filter_df(df, 1, 100)
embeddings_1_20 = get_embeddings(df_top1_20)

Number of samples=100, got 25 task groups
Using device: mps


In [13]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap_1d(embeddings_1_20, df_top1_20, neighbor, color_label="task_id")

/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



/Users/davebrunner/Documents/repositories/text_embedding_visualization/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

